# Lakeside heating DSM — human source of truth

**This notebook is the human-facing proof that the shipped model works.**

Training labels come from **native EnergyPlus** runs of the site **Lakeside**
staged utility champion (`LAKESIDE_SITE_ROOT` → Creekside site disk; Lakeside =
client rename of the same twin). There is **no** BAS `physics_proxy` / bootstrap path.

1. **Proof load** — open the farm parquet and show provenance / IDF hashes
2. **Measured vs modeled** overlay (baseline twin vs site meters)
3. **Sklearn bake-off** → champion → desktop ONNX ship
4. **Recursive 24h walk** (deployment mode)

| | |
|---|---|
| **Desktop target** | `facility_kw` only |
| **Training labels** | Native E+ IdealLoads + fixed-COP site kW |
| **Desktop artifact** | `ml/artifacts/heating_dsm_hourly_v1.onnx` |
| **Honesty** | IdealLoads+COP · CANDIDATE · zero-severe native runs only |


## 0 · Setup


In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
ML = ROOT / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import artifact_paths, train_parquet_path
from feature_compile_heating_dsm import (
    FEATURE_COLS, compile_features, matrix_xy, morning_peak_mask,
    assert_no_future_leakage, cost_from_hourly_kw,
)
from train_heating_dsm import bake_off
from export_sklearn_onnx import ship_desktop_champion, DISPLAY_NAMES
from notebook_plots import (
    family_cv_mae_bars, family_mae_rmse_grouped, leaderboard_table,
    oat_vs_kw_scatter, strategy_morning_peak_bars, example_day_profiles,
    residual_hist, pred_vs_actual, feature_importance_bar,
    explainer_vs_target_grid, lag_dependence_panel, metrics_scorecard,
    feature_target_catalogs, save_fig,
)

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
import os
from IPython.display import display, Markdown
from notebook_proof import prove_native_farm_load
SITE = Path(os.environ.get("LAKESIDE_SITE_ROOT", r"C:\Users\ben\OneDrive\Desktop\testing\sp_creekside"))
print("ROOT", ROOT)
print("SITE", SITE)
print("features", len(FEATURE_COLS))


## Proof — load native EnergyPlus farm (human visible)


In [ ]:
df_raw, proof = prove_native_farm_load(root=ROOT, paths=PATHS, site=SITE)
assert train_parquet_path().resolve() == PATHS["eplus_farm"].resolve()
print("train_parquet_path OK →", train_parquet_path())



## Compile features


In [ ]:
feat = compile_features(df_raw)
assert_no_future_leakage(feat)
X, y, groups, cols = matrix_xy(feat)
peak = morning_peak_mask(feat)
df = feat
print(f"rows={len(feat)} days={feat['day'].nunique()} features={len(cols)} peak_hours={int(peak.sum())}")
show = [c for c in ("day", "hour_ending", "facility_kw", "oat_f", "strategy_id", "provenance") if c in feat.columns]
display(feat[show].head(12))
PQ = PATHS['eplus_farm']
print('PQ', PQ)



## 1 · Explainer features & prediction target

Lag columns are marked **explainer (LAG)** — same-calendar-day only (no future leakage).


In [ ]:
feat_cat, tgt_cat = feature_target_catalogs()
print(f"{len(feat_cat)} explainer features · {len(tgt_cat)} target(s)")
display(feat_cat)
display(tgt_cat)
lag_rows = feat_cat[feat_cat["role"].str.contains("LAG")]
print("LAG features in the matrix:")
display(lag_rows)


## 2 · Load train parquet (E+ farm preferred)


## 3 · Feature compile + leakage guard


In [ ]:
feat = compile_features(df)
assert_no_future_leakage(df)
X, y, groups, cols = matrix_xy(df)
peak = morning_peak_mask(df)
print("X", X.shape, "peak hours", int(peak.sum()), "days", pd.Series(groups).nunique())
feat[FEATURE_COLS].describe().T.head(12)


## 4 · EDA — weather, strategies, **explainers vs target**, lag dependence


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
oat_vs_kw_scatter(df, ax=axes[0])
strategy_morning_peak_bars(df, ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "oat_and_strategy_peak.png", fig)
plt.show()

cold = (
    df[df["is_weekend"] < 0.5]
    .groupby("day")["oat_f"].mean()
    .sort_values()
    .index[0]
)
fig, ax = plt.subplots(figsize=(9, 4))
example_day_profiles(df, cold, ax=ax)
save_fig(PATHS["figures"] / "example_cold_day_strategies.png", fig)
plt.show()
print("example day", cold)

fig = explainer_vs_target_grid(df)
if fig is not None:
    plt.tight_layout()
    save_fig(PATHS["figures"] / "explainer_vs_target.png", fig)
    plt.show()

fig, ax = plt.subplots(figsize=(8, 3.8))
lag_dependence_panel(y, X, cols, peak, ax=ax)
save_fig(PATHS["figures"] / "lag_dependence.png", fig)
plt.show()


## 5 · Model bake-off (GroupKFold) — MAE / RMSE leaderboard

~2× wider RandomizedSearch grids per family (Ridge, ElasticNet, RF, GradientBoosting, ExtraTrees, HGB).
**Desktop ships the bake-off champion** (best morning-peak MAE), exported via skl2onnx.


In [ ]:
result = bake_off(df, n_splits=4, n_iter=40, n_iter_extra_trees=80)
lb = leaderboard_table(result["leaderboard"], result["cv"]["persistence"])
display(lb)
print("CHAMPION (desktop ship):", result["champion"])
print("  display:", DISPLAY_NAMES.get(result["champion"], result["champion"]))
print("  beat persistence:", result["beat_persistence_peak"])
print("  best_params:", result.get("best_params"))
print("  peak MAE:", result["cv"][result["champion"]]["mae_peak_05_09"])
print("  search iters:", result.get("search_iters"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
family_cv_mae_bars(result["leaderboard"], result["cv"]["persistence"]["mae_peak_05_09"], ax=axes[0])
family_mae_rmse_grouped(result["leaderboard"], result["cv"]["persistence"], ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "sklearn_leaderboard.png", fig)
plt.show()


## 6 · Champion OOF scorecard — pred vs actual, residuals, importances

Competition-style **MAE / RMSE / R²** on all hours and peak HE 05–09.


In [ ]:
model = result["model"]
oof = np.zeros_like(y)
gkf = GroupKFold(n_splits=result["n_splits"])
for tr, te in gkf.split(X, y, groups):
    m = clone(model)
    m.fit(X[tr], y[tr])
    oof[te] = m.predict(X[te])

# persistence baseline for scorecard
lag_i = cols.index("facility_kw_lag1")
persist = X[:, lag_i]

score = pd.concat(
    [
        metrics_scorecard(y, oof, peak, label=result["champion"]),
        metrics_scorecard(y, persist, peak, label="persistence_lag1"),
    ],
    ignore_index=True,
)
display(score)
print(
    "Lift vs lag1 on peak MAE:",
    float(score.loc[(score.model == "persistence_lag1") & (score.split == "peak_HE_05_09"), "mae"].iloc[0]
          - score.loc[(score.model == result["champion"]) & (score.split == "peak_HE_05_09"), "mae"].iloc[0]),
    "kW",
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pred_vs_actual(y, oof, ax=axes[0], title=f"OOF pred vs actual — {result['champion']}")
residual_hist(y, oof, ax=axes[1])
feature_importance_bar(model, cols, top_n=15, ax=axes[2])
plt.tight_layout()
save_fig(PATHS["figures"] / "oof_parity_resid_importance.png", fig)
plt.show()

# Peak-only parity
fig, ax = plt.subplots(figsize=(5, 4.5))
pred_vs_actual(y[peak], oof[peak], ax=ax, title="OOF pred vs actual — morning peak only")
save_fig(PATHS["figures"] / "oof_parity_peak.png", fig)
plt.show()


## 7 · Cost playground (bill-rate placeholders)

Same formula as Rust desktop: \(c_e \sum kWh + c_d \cdot peak\,kW\).


In [ ]:
demo_day = cold
rates = {
    "energy_rate_per_kwh": 0.12,
    "demand_rate_per_kw": 15.0,
    "similar_days_per_year": 90.0,
}

rows = []
for sid in ["baseline", "stagger_preheat", "flat_24_7", "morning_all_on"]:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, yd, _, _ = matrix_xy(sub)
    pred = model.predict(Xd)
    cost = cost_from_hourly_kw(pred, **rates)
    cost["strategy_id"] = sid
    cost["true_peak"] = float(sub["facility_kw"].max())
    rows.append(cost)

cost_df = pd.DataFrame(rows).set_index("strategy_id")
display(cost_df[["energy_kwh", "peak_kw", "energy_cost", "demand_cost", "total_cost", "annual_total_stub"]])

fig, ax = plt.subplots(figsize=(9, 4))
for sid in cost_df.index:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, _, _, _ = matrix_xy(sub)
    ax.plot(sub["hour_ending"], model.predict(Xd), label=sid, lw=1.8)
ce, cd = rates["energy_rate_per_kwh"], rates["demand_rate_per_kw"]
ax.set_title(f"Model 24h profiles — {demo_day} (${ce}/kWh + ${cd}/kW)")
ax.set_xlabel("Hour local")
ax.set_ylabel("pred facility_kw")
ax.legend(fontsize=8, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
save_fig(PATHS["figures"] / "forecast_day_cost_profiles.png", fig)
plt.show()


## 8 · Ship bake-off champion → desktop ONNX + TL;DR

Exports **whatever family won** the peak-MAE bake-off to:

- heating_dsm_hourly_v1.onnx + _feature_meta.json (name, params, MAE/RMSE, ±)
- joblib bundle + champion_summary.json
- copy → desktop/artifacts/ for cargo run --release

If skl2onnx cannot convert the winner, the next-best convertible family is shipped (noted in meta).


In [ ]:
PATHS["joblib"].parent.mkdir(parents=True, exist_ok=True)

champ = result["champion"]
joblib.dump(
    {
        "model": result["model"],
        "tuned_models": result["tuned_models"],
        "feature_cols": result["feature_cols"],
        "champion": champ,
        "best_params": result.get("best_params"),
        "best_params_by_family": result["best_params_by_family"],
        "schema": "lakeside.heating_dsm_hourly.v1",
    },
    PATHS["joblib"],
)

ship = ship_desktop_champion(
    result,
    onnx_path=PATHS["onnx"],
    meta_path=PATHS["feature_meta"],
    training_source=src,
)
summary = {
    "champion": champ,
    "desktop_onnx_family": ship["desktop_family"],
    "model_name": ship["model_name"],
    "best_params": ship["best_params"],
    "desktop_cv": ship["cv"],
    "beat_persistence_peak": result["beat_persistence_peak"],
    "training_source": src,
    "training_parquet": str(pq),
    "oof_scorecard": score.to_dict(orient="records"),
    "cv": result["cv"],
    "onnx_roundtrip_max_abs": ship["roundtrip_max_abs"],
    "fallback_from": ship.get("fallback_from"),
    "leaderboard": [
        {"family": e["family"], "oof_metrics": e["oof_metrics"], "n_iter_searched": e.get("n_iter_searched")}
        for e in result["leaderboard"]
    ],
}
PATHS["champion_summary"].write_text(json.dumps(summary, indent=2, default=str) + "\n", encoding="utf-8")

print("DESKTOP SHIP")
print("  bake-off champion:", champ)
print("  onnx family:", ship["desktop_family"], ship["model_name"])
print("  params:", ship["best_params"])
print("  cv:", ship["cv"])
print("  precision_pm_kw:", ship["meta"].get("precision_pm_kw"))
print("  onnx:", PATHS["onnx"])
print("  copy:", ship["desktop_copy"])
print("  roundtrip_max_abs:", ship["roundtrip_max_abs"])
if ship.get("fallback_from"):
    print("  NOTE: ONNX fallback from", ship["fallback_from"])
print("wrote", PATHS["joblib"])
print("wrote", PATHS["champion_summary"])


## 10 · Multi-target catalog + GroupKFold scorecard

`FEATURE_COLS_MULTITARGET` = single-target features + `zone_temp_*_f_lag1`.
Targets: `TARGET_COLS` = `facility_kw` + 6 zone temps.


In [ ]:
feat_cat_mt, tgt_cat_mt = feature_target_catalogs(multitarget=True)
print(f"{len(feat_cat_mt)} explainers · {len(tgt_cat_mt)} targets")
display(tgt_cat_mt)
display(feat_cat_mt[feat_cat_mt["role"].astype(str).str.contains("LAG|zone", case=False, na=False)].head(20))

X_mt, Y_mt, groups_mt, feat_cols_mt, tgt_cols_mt = matrix_xy_multi(df_mt)
peak_mt = morning_peak_mask(df_mt)
print("X", X_mt.shape, "Y", Y_mt.shape)


In [ ]:
gkf_mt = GroupKFold(n_splits=min(4, max(2, df_mt["day"].nunique())))
oof_mt = np.zeros_like(Y_mt)
for tr, te in gkf_mt.split(X_mt, Y_mt, groups_mt):
    est = MultiOutputRegressor(
        ExtraTreesRegressor(
            n_estimators=120, max_depth=14, min_samples_leaf=2,
            n_jobs=-1, random_state=21,
        )
    )
    est.fit(X_mt[tr], Y_mt[tr])
    oof_mt[te] = est.predict(X_mt[te])

per_target = {}
for j, name in enumerate(tgt_cols_mt):
    yt, yp = Y_mt[:, j], oof_mt[:, j]
    per_target[name] = {
        "mae": float(mean_absolute_error(yt, yp)),
        "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        "r2": float(r2_score(yt, yp)),
    }
# peak-window on facility_kw only
pm = peak_mt
per_target["facility_kw_peak_05_09"] = {
    "mae": float(mean_absolute_error(Y_mt[pm, 0], oof_mt[pm, 0])),
    "rmse": float(np.sqrt(mean_squared_error(Y_mt[pm, 0], oof_mt[pm, 0]))),
    "r2": float(r2_score(Y_mt[pm, 0], oof_mt[pm, 0])),
}
score_mt = pd.DataFrame(per_target).T.round(3)
display(score_mt)

fig, ax = plt.subplots(figsize=(8, 5))
bars = {k: v for k, v in per_target.items() if not k.endswith("peak_05_09")}
multitarget_mae_rmse_bars(bars, ax=ax)
save_fig(PATHS["figures"] / "multitarget_sklearn_mae_rmse.png", fig)
plt.show()
